# IndicF5 fine-tune — GPU pre-flight (G1–G14)

Runs FINETUNE_PLAN §8b's checks on 2×T4 against the code on the `fine-tune` branch. Nothing here trains a
model you keep; it proves the trainer, the data and the checkpoint path before any GPU hour goes to training.

**Before running:**
1. Settings → Accelerator: **GPU T4 x2**. Internet: **on**.
2. Add-ons → Secrets: `HF_TOKEN`, a Hugging Face token with **write** access (G14 uploads checkpoints) and
   access to `ai4bharat/IndicF5`.
3. The pre-flight dataset must be on the Hub as the private dataset repo `ayushk1233/indicf5-finetune-preflight`,
   with `route_a/`, `route_b/` and `audio/` **at the repo root**. From the repo on the machine that built it:

   `hf upload ayushk1233/indicf5-finetune-preflight finetune_data/preflight . --repo-type dataset --private --exclude "rows.jsonl" --exclude "xlit.json"`

4. Optional: the private model repo `ayushk1233/indicf5-finetune-ckpt` for G14's checkpoints. If it does not
   exist, G14 falls back to a local store and is reported as partial.

**Two passes:**
- **Interactive session, "Run All"** — G1–G13 (about 3–4 h). G14 is skipped.
- **"Save Version → Save & Run All"** (a committed background run) — only G14, the 20-minute committed run
  with the time guard at 12 minutes (§6a). G1–G13 are skipped in that mode.

Each pass ends by writing `preflight_gpu_report.md` and the per-check `g*.json` files under
`/kaggle/working/preflight` — download that folder and hand it back.

In [ ]:
import os
BATCH = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch"      # committed run: G14 only
print("mode:", "committed (G14 only)" if BATCH else "interactive (G1-G13)")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

## 1. Token (never printed)

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF_TOKEN set:", bool(os.environ["HF_TOKEN"]))

## 2. Code — the `fine-tune` branch

In [ ]:
os.chdir("/kaggle/working")
!rm -rf indic-dub-pipeline
!git clone -q -b fine-tune https://github.com/ayushk1233/indic-dub-pipeline.git
os.chdir("/kaggle/working/indic-dub-pipeline")
!git log --oneline -1
REPO = "/kaggle/working/indic-dub-pipeline"
VT = "/kaggle/tmp/vt/bin/python"
BASE = "/kaggle/tmp/indicf5/model.safetensors"
DATA = "/kaggle/tmp/data"
OUT = "/kaggle/working/preflight"
os.makedirs(OUT, exist_ok=True)

## 3. Two environments (§6d)
- **train venv** at `/kaggle/tmp/vt`: torch 2.11.0+cu126 and the training pins. Every check runs here.
- **shipping stack** in Kaggle's own Python: IndicF5 + `requirements-gpu.txt`, exactly as the dubbing notebook
  installs it. Used only by `official_mel.py` (G3, G13), always as a subprocess, so no restart is needed.

About 5 minutes.

In [ ]:
!python -m venv /kaggle/tmp/vt
!{VT} -m pip install -q --upgrade pip
!{VT} -m pip install -q -r requirements-train-kaggle.txt 2>&1 | tail -3
!{VT} -c "import torch, numpy, transformers, accelerate, peft; print(torch.__version__, torch.version.cuda, torch.cuda.device_count(), numpy.__version__, transformers.__version__, accelerate.__version__, peft.__version__)"
!pip install -q git+https://github.com/ai4bharat/IndicF5.git 2>&1 | tail -2
!pip install -q -r requirements-gpu.txt 2>&1 | tail -2

## 4. Pinned base weights and the pre-flight dataset

In [ ]:
!{VT} -c "import os; from huggingface_hub import snapshot_download; snapshot_download('ai4bharat/IndicF5', revision='ba85abedf18dc479a447eaa0eccbd76ab78a47d5', local_dir='/kaggle/tmp/indicf5', allow_patterns=['model.safetensors', 'checkpoints/vocab.txt'], token=os.environ['HF_TOKEN'])"
got = !sha256sum {BASE}
assert got[0].split()[0] == "ba7f3671180fb7784e24bd1dafc96e729a38ce02e7f6d3877cdef32525a1865c", got
print("base weights: sha256 ok")
!{VT} -c "import os; from huggingface_hub import snapshot_download; snapshot_download('ayushk1233/indicf5-finetune-preflight', repo_type='dataset', local_dir='/kaggle/tmp/data', token=os.environ['HF_TOKEN'])"
for route in ("route_a", "route_b"):
    for split in ("train", "val_en", "overfit16"):
        assert os.path.isdir(f"{DATA}/{route}/{split}/raw"), (route, split)
assert os.path.isdir(f"{DATA}/audio")
print("dataset ok")

## 5. Checks, in §8b order
Each cell prints its verdict and writes `OUT/gN.json`. A failing check does not stop the notebook; the report
at the end lists every result.

**G1 — environment** (10 min): 2 GPUs, fp16 autocast, NCCL all_reduce < 10 s.

In [ ]:
if not BATCH:
    !{VT} -m train.preflight_runners g1 --out {OUT}

**G2 — weight load** (15 min): strict load, §0 module names, parameter count, nothing on meta.

In [ ]:
if not BATCH:
    !{VT} -m train.preflight_runners g2 --base {BASE} --out {OUT}

**G3 — equivalence with the shipping path** (15 min): the same request through `colab.runtime.load_indicf5()`
(shipping stack) and through our loader + `train.generate` (train venv), fp32; mel max abs diff < 1e-3.

In [ ]:
if not BATCH:
    !{VT} -m train.preflight_runners request --data {DATA} --out {OUT}
    import json
    req = json.load(open(f"{OUT}/request.json"))
    !python train/kaggle/official_mel.py --ref "{req['ref']}" --ref-text "{req['ref_text']}" --gen-text "{req['gen_text']}" --seed {req['seed']} --fix-duration {req['fix_duration']} --out {OUT}/official_g3.npz
    !{VT} -m train.preflight_runners g3 --base {BASE} --official {OUT}/official_g3.npz --out {OUT}

**G4 — mel parity** (10 min): the training mel equals the inference mel for the same file.

In [ ]:
if not BATCH:
    !{VT} -m train.preflight_runners g4 --data {DATA} --out {OUT}

**G5 — baseline losses** (10 min): untouched model on `val_en`, both transcript forms. Partial until `val_indic` exists.

In [ ]:
if not BATCH:
    !{VT} -m train.preflight_runners g5 --base {BASE} --data {DATA} --out {OUT}

**G7 — memory, per route** (40 min): largest frames per GPU at ≤ 85% of 16 GB with 20 s clips. Runs before G6 and G8 so they use its number.

In [ ]:
if not BATCH:
    !{VT} -m train.preflight_runners g7 --route a --base {BASE} --out {OUT}
    !{VT} -m train.preflight_runners g7 --route b --base {BASE} --out {OUT}

**G6 — overfit, per route** (35 min): 16 clips, 300 updates, loss must fall > 50%; one sentence saved per route for scoring.

In [ ]:
if not BATCH:
    !{VT} -m train.preflight_runners g6 --route a --base {BASE} --data {DATA} --out {OUT}
    !{VT} -m train.preflight_runners g6 --route b --base {BASE} --data {DATA} --out {OUT}

**G8 — throughput** (20 min): 200 steady updates on 2 GPUs; data wait < 10%.

In [ ]:
if not BATCH:
    !{VT} -m train.preflight_runners g8 --base {BASE} --data {DATA} --out {OUT}

**G9 — DDP scaling** (20 min): single GPU at equal effective batch; 2 GPUs ≥ 1.7× and matching loss.

In [ ]:
if not BATCH:
    !{VT} -m train.preflight_runners g9 --base {BASE} --data {DATA} --out {OUT}

**G10 — fp16 stability** (40–60 min): 1,000 updates, no NaN/inf, GradScaler healthy.

In [ ]:
if not BATCH:
    !{VT} -m train.preflight_runners g10 --base {BASE} --data {DATA} --out {OUT}

**G11 — real resume on 2 GPUs** (15 min): killed at update 150, relaunched, continues from 100 to 250.

In [ ]:
if not BATCH:
    !{VT} -m train.preflight_runners g11 --base {BASE} --data {DATA} --out {OUT}

**G12 — validation cost** (from G10's log): < 5% of training time.

In [ ]:
if not BATCH:
    !{VT} -m train.preflight_runners g12 --out {OUT}

**G13 — export end to end** (15 min): merge G6's Route A LoRA into the base, load the merged file through the
shipping loader, and compare with the training-side model on the G3 request.

In [ ]:
if not BATCH:
    !{VT} -m train.preflight_runners g13 --base {BASE} --out {OUT}
    !python train/kaggle/official_mel.py --ref "{req['ref']}" --ref-text "{req['ref_text']}" --gen-text "{req['gen_text']}" --seed {req['seed']} --fix-duration {req['fix_duration']} --weights {OUT}/g13_merged.safetensors --out {OUT}/official_g13.npz
    !{VT} -m train.preflight_runners g13 --base {BASE} --official {OUT}/official_g13.npz --out {OUT}

**G14 — committed run** (25 min, committed mode only): a real session with the time guard at 12 minutes;
checkpoints upload to the private Hub repo, `LATEST` is written last, the notebook exits cleanly.

In [ ]:
if BATCH:
    from huggingface_hub import HfApi
    CKPT = "ayushk1233/indicf5-finetune-ckpt"
    if HfApi(token=os.environ["HF_TOKEN"]).repo_exists(CKPT):
        !{VT} -m train.preflight_runners g14 --base {BASE} --data {DATA} --hub-repo {CKPT} --out {OUT}
    else:
        print(CKPT, "does not exist: G14 runs against a local store (partial)")
        !{VT} -m train.preflight_runners g14 --base {BASE} --data {DATA} --out {OUT}

## 6. Report

In [ ]:
!{VT} -m train.preflight_runners report --out {OUT}
!cp train/preflight_gpu_report.md {OUT}/
!ls {OUT}